In [1]:
from pathlib import Path
import os
import re
import time
import html
import unicodedata

import joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.metrics import matthews_corrcoef, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
print("Environment ready")


Environment ready


In [2]:
BASE_EXPERIMENT_TAG = "linearsvc_binarycount_nospacechar_charwb2to5_word_tfidf"
DEFAULT_C = 0.25
C_VALUES = [0.10, 0.20, 0.25, 0.35, 0.50, 0.75, 1.00]

BEST_C = DEFAULT_C
EXPERIMENT_TAG = f"{BASE_EXPERIMENT_TAG}_C{str(BEST_C).replace('.', 'p')}_rs42"
CLASSIFIER = LinearSVC(C=BEST_C, random_state=RANDOM_STATE, dual="auto", max_iter=3000)
print("Experiment:", EXPERIMENT_TAG)
print("Classifier:", CLASSIFIER)
print("C candidates:", C_VALUES)


Experiment: linearsvc_binarycount_nospacechar_charwb2to5_word_tfidf_C0p25_rs42
Classifier: LinearSVC(C=0.25, max_iter=3000, random_state=42)
C candidates: [0.1, 0.2, 0.25, 0.35, 0.5, 0.75, 1.0]


In [3]:
# Colab default: put public_train.csv and public_test.csv in /content/drive/MyDrive/UD_26
# Local default: this repository keeps the files in ./mid

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

DATA_DIR_CANDIDATES = []
if os.environ.get("DATA_DIR"):
    DATA_DIR_CANDIDATES.append(Path(os.environ["DATA_DIR"]))

cwd = Path.cwd()
DATA_DIR_CANDIDATES.extend([
    Path("/content/drive/MyDrive/UD_26"),
    Path("/content/drive/MyDrive/UD_26/mid"),
    cwd,
    cwd / "mid",
    cwd / "data",
    cwd.parent,
    cwd.parent / "mid",
])

seen = set()
DATA_DIR_CANDIDATES = [p for p in DATA_DIR_CANDIDATES if not (str(p) in seen or seen.add(str(p)))]

DATA_DIR = None
for candidate in DATA_DIR_CANDIDATES:
    if (candidate / "public_train.csv").exists() and (candidate / "public_test.csv").exists():
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    checked = "\n".join(str(p) for p in DATA_DIR_CANDIDATES)
    raise FileNotFoundError(
        "Could not find public_train.csv and public_test.csv.\n"
        "Colab: run this after Drive is mounted and place the files in /content/drive/MyDrive/UD_26 or /content/drive/MyDrive/UD_26/mid.\n"
        "Local: run from the UD_26 repository root or from the mid folder, or set DATA_DIR to the folder containing the CSV files.\n"
        f"Current working directory: {cwd}\n"
        f"Checked paths:\n{checked}"
    )

TRAIN_PATH = DATA_DIR / "public_train.csv"
TEST_PATH = DATA_DIR / "public_test.csv"
OUTPUT_DIR = DATA_DIR
MODEL_DIR = OUTPUT_DIR / "model" / EXPERIMENT_TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_MODEL_PATH = MODEL_DIR / f"{EXPERIMENT_TAG}.pkl"
FINAL_MODEL_PATH = MODEL_DIR / "final_pipeline.pkl"
SUBMISSION_PATH = MODEL_DIR / "submission.csv"

print("IN_COLAB:", IN_COLAB)
print("Current working directory:", cwd)
print("DATA_DIR:", DATA_DIR.resolve())
print("TRAIN_PATH:", TRAIN_PATH.resolve())
print("TEST_PATH:", TEST_PATH.resolve())
print("MODEL_DIR:", MODEL_DIR.resolve())


IN_COLAB: True
Current working directory: /content
DATA_DIR: /content/drive/MyDrive/UD_26
TRAIN_PATH: /content/drive/MyDrive/UD_26/public_train.csv
TEST_PATH: /content/drive/MyDrive/UD_26/public_test.csv
MODEL_DIR: /content/drive/MyDrive/UD_26/model/linearsvc_binarycount_nospacechar_charwb2to5_word_tfidf_C0p25_rs42


In [4]:
train = pd.read_csv(TRAIN_PATH, encoding="utf-8")
test = pd.read_csv(TEST_PATH, encoding="utf-8")

print("train shape:", train.shape)
print("test shape:", test.shape)
print("label counts:")
print(train["label"].value_counts())

assert list(train.columns) == ["row_id", "text", "label"]
assert list(test.columns) == ["row_id", "text"]
assert set(train["label"].unique()) == {"POSITIVE", "NEGATIVE"}
assert train["text"].isna().sum() == 0
assert test["text"].isna().sum() == 0

X = train["text"].astype(str)
y = train["label"].astype(str)
X_test = test["text"].astype(str)


train shape: (149995, 3)
test shape: (49997, 2)
label counts:
label
NEGATIVE    75170
POSITIVE    74825
Name: count, dtype: int64


In [5]:
def preprocess_raw_texts(X):
    cleaned = []
    for x in X:
        x = "" if x is None else str(x)
        x = unicodedata.normalize("NFKC", html.unescape(x))
        x = re.sub(r"<[^>]+>", " ", x)
        x = re.sub(r"https?://\S+|www\.\S+", " URL ", x)
        x = re.sub(r"\s+", " ", x).strip().lower()
        cleaned.append(x)
    return cleaned


def preprocess_nospace_texts(X):
    return [re.sub(r"\s+", "", x) for x in preprocess_raw_texts(X)]


In [6]:
def build_features():
    return Pipeline([
        ("features", FeatureUnion([
            ("nospace_char", Pipeline([
                ("prep", FunctionTransformer(preprocess_nospace_texts, validate=False)),
                ("vec", CountVectorizer(
                    analyzer="char",
                    ngram_range=(1, 6),
                    min_df=2,
                    max_features=800_000,
                    lowercase=False,
                    binary=True,
                    dtype=np.float32,
                )),
            ])),
            ("raw_char_wb", Pipeline([
                ("prep", FunctionTransformer(preprocess_raw_texts, validate=False)),
                ("vec", CountVectorizer(
                    analyzer="char_wb",
                    ngram_range=(2, 5),
                    min_df=2,
                    max_features=500_000,
                    lowercase=False,
                    binary=True,
                    dtype=np.float32,
                )),
            ])),
            ("raw_word", Pipeline([
                ("prep", FunctionTransformer(preprocess_raw_texts, validate=False)),
                ("vec", CountVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=300_000,
                    token_pattern=r"(?u)\b\w+\b",
                    lowercase=False,
                    binary=True,
                    dtype=np.float32,
                )),
            ])),
        ], n_jobs=1)),
        ("tfidf", TfidfTransformer(
            sublinear_tf=False,
            norm="l2",
            use_idf=True,
            smooth_idf=True,
        )),
    ])


def build_pipeline():
    return Pipeline([
        ("features", build_features()),
        ("classifier", CLASSIFIER),
    ])

pipeline = build_pipeline()
pipeline


Pipeline(steps=[('features',
                 Pipeline(steps=[('features',
                                  FeatureUnion(n_jobs=1,
                                               transformer_list=[('nospace_char',
                                                                  Pipeline(steps=[('prep',
                                                                                   FunctionTransformer(func=<function preprocess_nospace_texts at 0x7ca9045a98a0>)),
                                                                                  ('vec',
                                                                                   CountVectorizer(analyzer='char',
                                                                                                   binary=True,
                                                                                                   dtype=<class 'numpy.float32'>,
                                                                                                   lowercase=False,
                                                                                                   max_features=800000,
                                                                                                   min_df=2,
                                                                                                   ngram_range...
                                                                  Pipeline(steps=[('prep',
                                                                                   FunctionTransformer(func=<function preprocess_raw_texts at 0x7ca9045a9800>)),
                                                                                  ('vec',
                                                                                   CountVectorizer(binary=True,
                                                                                                   dtype=<class 'numpy.float32'>,
                                                                                                   lowercase=False,
                                                                                                   max_features=300000,
                                                                                                   min_df=2,
                                                                                                   ngram_range=(1,
                                                                                                                2),
                                                                                                   token_pattern='(?u)\\b\\w+\\b'))]))])),
                                 ('tfidf', TfidfTransformer())])),
                ('classifier',
                 LinearSVC(C=0.25, max_iter=3000, random_state=42))])

In [ ]:
RUN_TUNING = True

if RUN_TUNING:
    start = time.time()
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    mcc_scorer = make_scorer(matthews_corrcoef)
    tuning_rows = []

    for c_value in C_VALUES:
        candidate = LinearSVC(C=c_value, random_state=RANDOM_STATE, dual="auto", max_iter=3000)
        CLASSIFIER = candidate
        scores = cross_val_score(
            build_pipeline(),
            X,
            y,
            scoring=mcc_scorer,
            cv=cv,
            n_jobs=1,
        )
        tuning_rows.append({
            "C": c_value,
            "mean_mcc": scores.mean(),
            "std_mcc": scores.std(),
            "fold_scores": scores,
        })
        print(f"C={c_value}: mean={scores.mean():.6f}, std={scores.std():.6f}, scores={scores}")

    tuning_results = pd.DataFrame(tuning_rows).sort_values("mean_mcc", ascending=False).reset_index(drop=True)
    BEST_C = float(tuning_results.loc[0, "C"])
    CLASSIFIER = LinearSVC(C=BEST_C, random_state=RANDOM_STATE, dual="auto", max_iter=3000)
    EXPERIMENT_TAG = f"{BASE_EXPERIMENT_TAG}_C{str(BEST_C).replace('.', 'p')}_rs42"
    MODEL_DIR = OUTPUT_DIR / "model" / EXPERIMENT_TAG
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    EXPERIMENT_MODEL_PATH = MODEL_DIR / f"{EXPERIMENT_TAG}.pkl"
    FINAL_MODEL_PATH = MODEL_DIR / "final_pipeline.pkl"
    SUBMISSION_PATH = MODEL_DIR / "submission.csv"

    print("\nTuning results:")
    print(tuning_results[["C", "mean_mcc", "std_mcc"]])
    print("Selected C:", BEST_C)
    print("Selected experiment:", EXPERIMENT_TAG)
    print(f"Elapsed: {time.time() - start:.1f} sec")
else:
    print("RUN_TUNING=False: using default C", BEST_C)


C=0.1: mean=0.746881, std=0.003260, scores=[0.75077538 0.74239172 0.74570521 0.74504074 0.75049167]
C=0.2: mean=0.755407, std=0.003614, scores=[0.75773277 0.74919118 0.75361155 0.75727818 0.75921946]
C=0.25: mean=0.757223, std=0.003272, scores=[0.75864037 0.75206182 0.7547997  0.76020816 0.76040282]
C=0.35: mean=0.758064, std=0.002854, scores=[0.76001697 0.75313186 0.75652627 0.76046374 0.76017929]
C=0.5: mean=0.758056, std=0.002678, scores=[0.75887403 0.7532036  0.75739109 0.7601262  0.7606863 ]


In [ ]:
start = time.time()
final_pipeline = build_pipeline()
final_pipeline.fit(X, y)

pred_test = final_pipeline.predict(X_test)
submission = pd.DataFrame({
    "row_id": test["row_id"],
    "pred_label": pred_test,
})

assert len(submission) == len(test)
assert set(submission["pred_label"].unique()) <= {"POSITIVE", "NEGATIVE"}

joblib.dump(final_pipeline, EXPERIMENT_MODEL_PATH, compress=3)
joblib.dump(final_pipeline, FINAL_MODEL_PATH, compress=3)
submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")

print("Saved experiment model:", EXPERIMENT_MODEL_PATH.resolve())
print("Saved final model:", FINAL_MODEL_PATH.resolve())
print("Saved submission:", SUBMISSION_PATH.resolve())
print("Prediction distribution:")
print(submission["pred_label"].value_counts())
print(f"Elapsed: {time.time() - start:.1f} sec")


In [ ]:
check = pd.read_csv(SUBMISSION_PATH, encoding="utf-8")
print(check.shape)
print(check.head())
print(check["pred_label"].value_counts())
assert list(check.columns) == ["row_id", "pred_label"]
assert len(check) == len(test)
assert check["row_id"].equals(test["row_id"])
assert set(check["pred_label"].unique()) <= {"POSITIVE", "NEGATIVE"}
print("Submission format OK")
